# 用 AI 辅助写代码和做研究：能做什么、必须检查什么

本笔记本由文章自动生成，代码和[原文](https://wxrkz123.github.io/blog/AI-Assisted-Research/)完全一致。文中的解释、公式、图和每段代码的输出，请对照原文阅读。

## 三、一个例子：看起来完全正确的代码

假设你让 AI 写一段代码：用股票的月度收益计算过去 3 个月的累计收益（动量信号），数据是长表格式，每行是一只股票一个月。它很可能给出这样的代码：

In [ ]:
import numpy as np
import pandas as pd

def momentum_ai(df):
    """计算过去 3 个月（含当月）的累计收益。"""
    df = df.sort_values(["stock", "month"]).copy()
    df["mom3"] = (1 + df["ret"]).rolling(3).apply(np.prod, raw=True) - 1
    return df

**用一个答案已知的小例子测试它**。构造两只股票、每只 4 个月的数据，手算出正确答案：

In [ ]:
test = pd.DataFrame({
    "stock": ["A"] * 4 + ["B"] * 4,
    "month": [1, 2, 3, 4] * 2,
    "ret":   [0.10, 0.10, 0.10, 0.10, 0.0, 0.0, 0.0, 0.0],
})
expected = [np.nan, np.nan, 1.1 ** 3 - 1, 1.1 ** 3 - 1, np.nan, np.nan, 0.0, 0.0]   # 手算的答案
out = momentum_ai(test)
print(out.assign(expected=expected).round(4).to_string(index=False))

正确的写法要在每只股票内部计算：

In [ ]:
def momentum(df):
    df = df.sort_values(["stock", "month"]).copy()
    df["mom3"] = (df.groupby("stock")["ret"]
                    .transform(lambda r: (1 + r).rolling(3).apply(np.prod, raw=True) - 1))
    return df

out = momentum(test)
print("与手算的答案一致：", np.allclose(out["mom3"], expected, equal_nan=True))